<a href="https://colab.research.google.com/github/ShumwayRobert1980/ComfyUI/blob/master/fast_stable_diffusion_AUTOMATIC1111.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Colab Pro notebook from https://github.com/TheLastBen/fast-stable-diffusion. [ComfyUI Colab](https://colab.research.google.com/github/TheLastBen/fast-stable-diffusion/blob/main/fast_stable_diffusion_ComfyUI.ipynb)**

In [49]:
#@markdown # Connect Google Drive
from google.colab import drive
from IPython.display import clear_output
import ipywidgets as widgets
import os

def inf(msg, style, wdth): inf = widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth));display(inf)
Shared_Drive = "" #@param {type:"string"}
#@markdown - Leave empty if you're not using a shared drive

print("[0;33mConnecting...")
drive.mount('/content/gdrive')

if Shared_Drive!="" and os.path.exists("/content/gdrive/Shareddrives"):
  mainpth="Shareddrives/"+Shared_Drive
else:
  mainpth="MyDrive"

clear_output()
inf('\u2714 Done','success', '50px')

#@markdown ---

Button(button_style='success', description='✔ Done', disabled=True, layout=Layout(min_width='50px'), style=But…

In [50]:
#@markdown # Install/Update AUTOMATIC1111 repo
from IPython.utils import capture
from IPython.display import clear_output
from subprocess import getoutput
import ipywidgets as widgets
import sys
import fileinput
import os
import time
import base64
import requests
from urllib.request import urlopen, Request
from urllib.parse import urlparse, parse_qs, unquote
from tqdm import tqdm
import six


blsaphemy=base64.b64decode(("ZWJ1aQ==").encode('ascii')).decode('ascii')

if not os.path.exists("/content/gdrive"):
  print('[1;31mGdrive not connected, using temporary colab storage ...')
  time.sleep(4)
  mainpth="MyDrive"
  !mkdir -p /content/gdrive/$mainpth
  Shared_Drive=""

if Shared_Drive!="" and not os.path.exists("/content/gdrive/Shareddrives"):
  print('[1;31mShared drive not detected, using default MyDrive')
  mainpth="MyDrive"

with capture.capture_output() as cap:
  def inf(msg, style, wdth): inf = widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth));display(inf)
  fgitclone = "git clone --depth 1"
  !git clone -q --depth 1 --branch main https://github.com/TheLastBen/diffusers
  %mkdir -p /content/gdrive/$mainpth/sd
  %cd /content/gdrive/$mainpth/sd
  !git clone -q --branch master https://github.com/AUTOMATIC1111/stable-diffusion-w$blsaphemy
  !mkdir -p /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/cache/
  os.environ['TRANSFORMERS_CACHE']=f"/content/gdrive/{mainpth}/sd/stable-diffusion-w"+blsaphemy+"/cache"
  os.environ['TORCH_HOME'] = f"/content/gdrive/{mainpth}/sd/stable-diffusion-w"+blsaphemy+"/cache"
  !mkdir -p /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/repositories
  !git clone https://github.com/AUTOMATIC1111/stable-diffusion-w$blsaphemy-assets /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/repositories/stable-diffusion-webui-assets

with capture.capture_output() as cap:
  %cd /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/
  !git reset --hard
  !git checkout master
  time.sleep(1)
  !rm webui.sh
  !git pull
clear_output()
inf('\u2714 Done','success', '50px')

#@markdown ---

Button(button_style='success', description='✔ Done', disabled=True, layout=Layout(min_width='50px'), style=But…

In [51]:
print('Cloning ComfyUI into the stable-diffusion-webui extension directory...')
%cd /content/gdrive/{mainpth}/sd/stable-diffusion-webui/extensions/sd-webui-comfyui
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content
print('ComfyUI cloning complete.')

Cloning ComfyUI into the stable-diffusion-webui extension directory...
/content/gdrive/MyDrive/sd/stable-diffusion-webui/extensions/sd-webui-comfyui
fatal: destination path 'ComfyUI' already exists and is not an empty directory.
/content
ComfyUI cloning complete.


In [52]:
#@markdown # Requirements

print('[1;32mInstalling requirements...')

with capture.capture_output() as cap:
  !rm -r /usr/local/lib/python3.12/dist-packages/gradio*
  %cd /content/
  !wget -q -i https://raw.githubusercontent.com/TheLastBen/fast-stable-diffusion/main/Dependencies/A1111.txt
  !dpkg -i *.deb
  if not os.path.exists('/content/gdrive/'+mainpth+'/sd/stablediffusion'):
    !tar -C /content/gdrive/$mainpth --zstd -xf sd_mrep.tar.zst
  !tar -C / --zstd -xf gcolabdeps.tar.zst
  !rm *.deb | rm *.zst | rm *.txt
  if not os.path.exists('gdrive/'+mainpth+'/sd/libtcmalloc/libtcmalloc_minimal.so.4'):
    %env CXXFLAGS=-std=c++14
    !wget -q https://github.com/gperftools/gperftools/releases/download/gperftools-2.5/gperftools-2.5.tar.gz && tar zxf gperftools-2.5.tar.gz && mv gperftools-2.5 gperftools
    !wget -q https://github.com/TheLastBen/fast-stable-diffusion/raw/main/AUTOMATIC1111_files/Patch
    %cd /content/gperftools
    !patch -p1 < /content/Patch
    !./configure --enable-minimal --enable-libunwind --enable-frame-pointers --enable-dynamic-sized-delete-support --enable-sized-delete --enable-emergency-malloc; make -j4
    !mkdir -p /content/gdrive/$mainpth/sd/libtcmalloc && cp .libs/libtcmalloc*.so* /content/gdrive/$mainpth/sd/libtcmalloc
    %env LD_PRELOAD=/content/gdrive/$mainpth/sd/libtcmalloc/libtcmalloc_minimal.so.4
    %cd /content
    !rm *.tar.gz Patch && rm -r /content/gperftools
  else:
    %env LD_PRELOAD=/content/gdrive/$mainpth/sd/libtcmalloc/libtcmalloc_minimal.so.4

  !pip uninstall jax -y
  !pip install wandb==0.15.12 pydantic==1.10.2 numpy==1.26 scipy==1.15.3 controlnet_aux --no-deps -qq
  !pip install diffusers accelerate -U --no-deps -qq
  !rm -r /usr/local/lib/python3.12/dist-packages/tensorflow*
  os.environ['PYTHONWARNINGS'] = 'ignore'
  !sed -i 's@text = _formatwarnmsg(msg)@text =\"\"@g' /usr/lib/python3.12/warnings.py
  !sed -i 's@from pytorch_lightning.loggers.wandb import WandbLogger  # noqa: F401@@g' /usr/local/lib/python3.12/dist-packages/pytorch_lightning/loggers/__init__.py
  !sed -i 's@from .mailbox import ContextCancelledError@@g' /usr/local/lib/python3.12/dist-packages/wandb/sdk/lib/retry.py
  !sed -i 's@raise ContextCancelledError("retry timeout")@print("retry timeout")@g' /usr/local/lib/python3.12/dist-packages/wandb/sdk/lib/retry.py
  !sed -i 's@globalns, localns, set()@globalns, localns, recursive_guard=set()@g' /usr/local/lib/python3.12/dist-packages/pydantic/typing.py

clear_output()
inf('\u2714 Done','success', '50px')

#@markdown ---

Button(button_style='success', description='✔ Done', disabled=True, layout=Layout(min_width='50px'), style=But…

In [53]:
#@markdown # Model Download/Load

import gdown
from gdown.download import get_url_from_gdrive_confirmation
import re

Use_Temp_Storage = True #@param {type:"boolean"}
#@markdown - If not, make sure you have enough space on your gdrive

#@markdown ---

Model_Version = "SDXL" #@param ["SDXL", "1.5", "v1.5 Inpainting", "V2.1-768px"]

#@markdown Or
PATH_to_MODEL = "/content/gdrive/MyDrive/_18v2.safetensors" #@param {type:"string"}
#@markdown - Insert the full path of your custom model or to a folder containing multiple models

#@markdown Or
MODEL_LINK = "" #@param {type:"string"}


def getsrc(url):
    parsed_url = urlparse(url)
    if parsed_url.netloc == 'civitai.com':
        src='civitai'
    elif parsed_url.netloc == 'drive.google.com':
        src='gdrive'
    elif parsed_url.netloc == 'huggingface.co':
        src='huggingface'
    else:
        src='others'
    return src

src=getsrc(MODEL_LINK)

def get_name(url, gdrive):
    if not gdrive:
        response = requests.get(url, allow_redirects=False)
        if "Location" in response.headers:
            redirected_url = response.headers["Location"]
            quer = parse_qs(urlparse(redirected_url).query)
            if "response-content-disposition" in quer:
                disp_val = quer["response-content-disposition"][0].split(";")
                for vals in disp_val:
                    if vals.strip().startswith("filename="):
                        filenm=unquote(vals.split("=", 1)[1].strip())
                        return filenm.replace("\"","")
    else:
        headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36"}
        lnk="https://drive.google.com/uc?id={id}&export=download".format(id=url[url.find("/d/")+3:url.find("/view")])
        res = requests.session().get(lnk, headers=headers, stream=True, verify=True)
        res = requests.session().get(get_url_from_gdrive_confirmation(res.text), headers=headers, stream=True, verify=True)
        content_disposition = six.moves.urllib_parse.unquote(res.headers["Content-Disposition"])
        filenm = re.search('attachment; filename="(.*?)"', content_disposition).groups()[0]
        return filenm


def dwn(url, dst, msg):
    file_size = None
    req = Request(url, headers={"User-Agent": "torch.hub"})
    u = urlopen(req)
    meta = u.info()
    if hasattr(meta, 'getheaders'):
        content_length = meta.getheaders("Content-Length")
    else:
        content_length = meta.get_all("Content-Length")
    if content_length is not None and len(content_length) > 0:
        file_size = int(content_length[0])

    with tqdm(total=file_size, disable=False, mininterval=0.5,
              bar_format=msg+' |{bar:20}| {percentage:3.0f}%') as pbar:
        with open(dst, "wb") as f:
            while True:
                buffer = u.read(8192)
                if len(buffer) == 0:
                    break
                f.write(buffer)
                pbar.update(len(buffer))
            f.close()


def sdmdls(ver, Use_Temp_Storage):

  if ver=='1.5':
    if Use_Temp_Storage:
      os.makedirs('/content/temp_models', exist_ok=True)
      model='/content/temp_models/v1-5-pruned-emaonly.safetensors'
    else:
      model='/content/gdrive/'+mainpth+'/sd/stable-diffusion-w'+blsaphemy+'/models/Stable-diffusion/v1-5-pruned-emaonly.safetensors'
    link='https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors'
  elif ver=='V2.1-768px':
    if Use_Temp_Storage:
      os.makedirs('/content/temp_models', exist_ok=True)
      model='/content/temp_models/v2-1_768-ema-pruned.safetensors'
    else:
      model='/content/gdrive/'+mainpth+'/sd/stable-diffusion-w'+blsaphemy+'/models/Stable-diffusion/v2-1_768-ema-pruned.safetensors'
    link='https://huggingface.co/stabilityai/stable-diffusion-2-1/resolve/main/v2-1_768-ema-pruned.safetensors'
  elif ver=='v1.5 Inpainting':
    if Use_Temp_Storage:
      os.makedirs('/content/temp_models', exist_ok=True)
      model='/content/temp_models/sd-v1-5-inpainting.ckpt'
    else:
      model='/content/gdrive/'+mainpth+'/sd/stable-diffusion-w'+blsaphemy+'/models/Stable-diffusion/sd-v1-5-inpainting.ckpt'
    link='https://huggingface.co/runwayml/stable-diffusion-inpainting/resolve/main/sd-v1-5-inpainting.ckpt'
  elif ver=='SDXL':
    if Use_Temp_Storage:
      os.makedirs('/content/temp_models', exist_ok=True)
      model='/content/temp_models/sd_xl_base_1.0.safetensors'
    else:
      model='/content/gdrive/'+mainpth+'/sd/stable-diffusion-w'+blsaphemy+'/models/Stable-diffusion/sd_xl_base_1.0.safetensors'
    link='https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors'

  if not os.path.exists(model):
    !gdown --fuzzy -O $model $link
    if os.path.exists(model):
      clear_output()
      inf('\u2714 Done','success', '50px')
    else:
      inf('\u2718 Something went wrong, try again','danger', "250px")
  else:
      clear_output()
      inf('\u2714 Model already exists','primary', '300px')

  return model


if (PATH_to_MODEL !=''):
  if os.path.exists(str(PATH_to_MODEL)):
    inf('\u2714 Using the trained model.','success', '200px')

  else:
      while not os.path.exists(str(PATH_to_MODEL)):
        inf('\u2718 Wrong path, use the colab file explorer to copy the path : ','danger', "400px")
        PATH_to_MODEL=input()
      if os.path.exists(str(PATH_to_MODEL)):
        inf('\u2714 Using the custom model.','success', '200px')

  model=PATH_to_MODEL

elif MODEL_LINK != "":

      if src=='civitai':
         modelname=get_name(MODEL_LINK, False)
         if Use_Temp_Storage:
            os.makedirs('/content/temp_models', exist_ok=True)
            model=f'/content/temp_models/{modelname}'
         else:
            model=f'/content/gdrive/{mainpth}/sd/stable-diffusion-w{blsaphemy}/models/Stable-diffusion/{modelname}'
         if not os.path.exists(model):
            dwn(MODEL_LINK, model, 'Downloading the custom model')
            clear_output()
         else:
            inf('\u2714 Model already exists','primary', '300px')
      elif src=='gdrive':
         modelname=get_name(MODEL_LINK, True)
         if Use_Temp_Storage:
            os.makedirs('/content/temp_models', exist_ok=True)
            model=f'/content/temp_models/{modelname}'
         else:
            model=f'/content/gdrive/{mainpth}/sd/stable-diffusion-w{blsaphemy}/models/Stable-diffusion/{modelname}'
         if not os.path.exists(model):
            gdown.download(url=MODEL_LINK, output=model, quiet=False, fuzzy=True)
            clear_output()
         else:
            inf('\u2714 Model already exists','primary', '300px')
      else:
         modelname=os.path.basename(MODEL_LINK)
         if Use_Temp_Storage:
            os.makedirs('/content/temp_models', exist_ok=True)
            model=f'/content/temp_models/{modelname}'
         else:
            model=f'/content/gdrive/{mainpth}/sd/stable-diffusion-w{blsaphemy}/models/Stable-diffusion/{modelname}'
         if not os.path.exists(model):
            gdown.download(url=MODEL_LINK, output=model, quiet=False, fuzzy=True)
            clear_output()
         else:
            inf('\u2714 Model already exists','primary', '700px')

      if os.path.exists(model) and os.path.getsize(model) > 1810671599:
        inf('\u2714 Model downloaded, using the custom model.','success', '300px')
      else:
        !rm model
        inf('\u2718 Wrong link, check that the link is valid','danger', "300px")

else:
  model=sdmdls(Model_Version, Use_Temp_Storage)

#@markdown ---

Button(button_style='success', description='✔ Using the trained model.', disabled=True, layout=Layout(min_widt…

In [54]:
import os

model_path = '/content/gdrive/MyDrive/_18v2.safetensors'
if os.path.exists(model_path):
    print(f'\u2714 Model file found at: {model_path}')
else:
    print(f'\u2718 Model file NOT found at: {model_path}')

✔ Model file found at: /content/gdrive/MyDrive/_18v2.safetensors


In [55]:
print(f'Listing contents of /content/gdrive/{mainpth}:')
!ls -F /content/gdrive/$mainpth

print('\nIf you see your model file listed above, please update the `PATH_to_MODEL` variable in the `Model Download/Load` cell (`p4wj_txjP3TC`) with the correct full path (e.g., `/content/gdrive/MyDrive/your_model.safetensors`) and re-run that cell, followed by the verification cell.')

Listing contents of /content/gdrive/MyDrive:
 _18v2.safetensors
'Adult Animated Interactive Book Creation.gdoc'
 AI_ART/
 calander.gdoc
'Choosing the Best NSFW Model.gsheet'
'Colab Notebooks'/
 ComfyUI_Colab.ipynb
 ComfyUI_Outputs/
'Google AI Studio'/
'Grok 4.2 Jailbreak EDEN X XANDER System Prompt.gdoc'
'help me create a GEMS so i can re-learn my class....gsheet'
'Here is what Grok gave me, incorporate theirs and....gdoc'
 InvokeAI/
 InvokeAI_Colab_Fixed.ipynb
 InvokeAI_Colab.ipynb
 jailbreak001.png
'Lie Detection Codex Compilation Plan.gdoc'
'Memory for LaserForge AR.gsheet'
 models/
'MUSIC 2025'/
 Opal/
'Professional Information & Behavioral Observation Template.gdoc'
 README_GCP_Chat_Setup.md.gdoc
'Recrystallization Of Meth.gdoc'
 run_raw_qwen.bat
'Saved from Chrome'/
 sd/
 SD_LoRAs/
 SD_Models/
 SD_Output/
 setup_gcp_chat.sh.gdoc
 setup_raw_qwen.bat
 setup_raw_qwen.bat.gdoc
'Start research.gdoc'
'The Anatomy of Deception: Forensic and Psychological Intelligence Protocols.gdoc'
'Un

In [56]:
import os

model_path = '/content/gdrive/MyDrive/_18v2.safetensors'
if os.path.exists(model_path):
    print(f'\u2714 Model file found at: {model_path}')
else:
    print(f'\u2718 Model file NOT found at: {model_path}')

✔ Model file found at: /content/gdrive/MyDrive/_18v2.safetensors


In [57]:
#@markdown # Download LoRA

LoRA_LINK = "" #@param {type:"string"}

if LoRA_LINK == "":
  inf('\u2714 Nothing to do','primary', '200px')
else:
  os.makedirs('/content/gdrive/'+mainpth+'/sd/stable-diffusion-w'+blsaphemy+'/models/Lora', exist_ok=True)

  src=getsrc(LoRA_LINK)

  if src=='civitai':
      modelname=get_name(LoRA_LINK, False)
      loramodel=f'/content/gdrive/{mainpth}/sd/stable-diffusion-w{blsaphemy}/models/Lora/{modelname}'
      if not os.path.exists(loramodel):
        dwn(LoRA_LINK, loramodel, 'Downloading the LoRA model '+modelname)
        clear_output()
      else:
        inf('\u2714 Model already exists','primary', '200px')
  elif src=='gdrive':
      modelname=get_name(LoRA_LINK, True)
      loramodel=f'/content/gdrive/{mainpth}/sd/stable-diffusion-w{blsaphemy}/models/Lora/{modelname}'
      if not os.path.exists(loramodel):
        gdown.download(url=LoRA_LINK, output=loramodel, quiet=False, fuzzy=True)
        clear_output()
      else:
        inf('\u2714 Model already exists','primary', '200px')
  else:
      modelname=os.path.basename(LoRA_LINK)
      loramodel=f'/content/gdrive/{mainpth}/sd/stable-diffusion-w{blsaphemy}/models/Lora/{modelname}'
      if not os.path.exists(loramodel):
        gdown.download(url=LoRA_LINK, output=loramodel, quiet=False, fuzzy=True)
        clear_output()
      else:
        inf('\u2714 Model already exists','primary', '200px')

  if os.path.exists(loramodel) :
    inf('\u2714 LoRA downloaded','success', '200px')
  else:
    inf('\u2718 Wrong link, check that the link is valid','danger', "300px")

#@markdown ---

Button(button_style='primary', description='✔ Nothing to do', disabled=True, layout=Layout(min_width='200px'),…

In [58]:
#@markdown # ControlNet
from torch.hub import download_url_to_file
from urllib.parse import urlparse
import re
from subprocess import run

XL_Model = "All" #@param [ "None", "All", "Canny", "Depth", "Sketch", "OpenPose", "Recolor"]

v1_Model = "All (21GB)" #@param [ "None", "All (21GB)", "Canny", "Depth", "Lineart", "MLSD", "Normal", "OpenPose", "Scribble", "Seg", "ip2p", "Shuffle", "Inpaint", "Softedge", "Lineart_Anime", "Tile", "T2iadapter_Models"]

v2_Model = "All" #@param [ "None", "All", "Canny", "Depth", "HED", "OpenPose", "Scribble"]

#@markdown - Download/update ControlNet extension and its models

def download(url, model_dir):

    filename = os.path.basename(urlparse(url).path)
    pth = os.path.abspath(os.path.join(model_dir, filename))
    if not os.path.exists(pth):
        print('Downloading: '+os.path.basename(url))
        download_url_to_file(url, pth, hash_prefix=None, progress=True)
    else:
      print(f"[1;32mThe model {filename} already exists[0m")


Canny='https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/diffusers_xl_canny_mid.safetensors'
Depth='https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/diffusers_xl_depth_mid.safetensors'
Sketch='https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/sai_xl_sketch_256lora.safetensors'
OpenPose='https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/thibaud_xl_openpose_256lora.safetensors'
Recolor='https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/sai_xl_recolor_128lora.safetensors'


with capture.capture_output() as cap:
  %cd /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/extensions
  if not os.path.exists('sd-w'+blsaphemy+'-controlnet'):
    !git clone https://github.com/Mikubill/sd-w$blsaphemy-controlnet.git
    %cd /content
  else:
    %cd sd-w$blsaphemy-controlnet
    !git reset --hard
    !git pull
    %cd /content

mdldir='/content/gdrive/'+mainpth+'/sd/stable-diffusion-w'+blsaphemy+'/extensions/sd-w'+blsaphemy+'-controlnet/models'
for filename in os.listdir(mdldir):
  if "_sd14v1" in filename:
    renamed = re.sub("_sd14v1", "-fp16", filename)
    os.rename(os.path.join(mdldir, filename), os.path.join(mdldir, renamed))

!wget -q -O CN_models.txt https://github.com/TheLastBen/fast-stable-diffusion/raw/main/AUTOMATIC1111_files/CN_models.txt
!wget -q -O CN_models_v2.txt https://github.com/TheLastBen/fast-stable-diffusion/raw/main/AUTOMATIC1111_files/CN_models_v2.txt
!wget -q -O CN_models_XL.txt https://github.com/TheLastBen/fast-stable-diffusion/raw/main/AUTOMATIC1111_files/CN_models_XL.txt


with open("CN_models.txt", 'r') as f:
  mdllnk = f.read().splitlines()
with open("CN_models_v2.txt", 'r') as d:
  mdllnk_v2 = d.read().splitlines()
with open("CN_models_XL.txt", 'r') as d:
  mdllnk_XL = d.read().splitlines()

!rm CN_models.txt CN_models_v2.txt CN_models_XL.txt


if XL_Model == "All":
  for lnk_XL in mdllnk_XL:
      download(lnk_XL, mdldir)
  clear_output()
  inf('\u2714 Done','success', '50px')

elif XL_Model == "None":
    pass
    clear_output()
    inf('\u2714 Done','success', '50px')

else:
  download(globals()[XL_Model], mdldir)
  clear_output()
  inf('\u2714 Done','success', '50px')


Canny='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_canny.pth'
Depth='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11f1p_sd15_depth.pth'
Lineart='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_lineart.pth'
MLSD='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_mlsd.pth'
Normal='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_normalbae.pth'
OpenPose='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_openpose.pth'
Scribble='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_scribble.pth'
Seg='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_seg.pth'
ip2p='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11e_sd15_ip2p.pth'
Shuffle='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11e_sd15_shuffle.pth'
Inpaint='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_inpaint.pth'
Softedge='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_softedge.pth'
Lineart_Anime='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15s2_lineart_anime.pth'
Tile='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11f1e_sd15_tile.pth'


with capture.capture_output() as cap:
  cfgnames=[os.path.basename(url).split('.')[0]+'.yaml' for url in mdllnk_v2]
  %cd /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/extensions/sd-w$blsaphemy-controlnet/models
  for name in cfgnames:
      run(['cp', 'cldm_v21.yaml', name])
  %cd /content

if v1_Model == "All (21GB)":
  for lnk in mdllnk:
      download(lnk, mdldir)
  clear_output()

elif v1_Model == "T2iadapter_Models":
  mdllnk=list(filter(lambda x: 't2i' in x, mdllnk))
  for lnk in mdllnk:
      download(lnk, mdldir)
  clear_output()

elif v1_Model == "None":
    pass
    clear_output()

else:
  download(globals()[v1_Model], mdldir)
  clear_output()

Canny='https://huggingface.co/thibaud/controlnet-sd21/resolve/main/control_v11p_sd21_canny.safetensors'
Depth='https://huggingface.co/thibaud/controlnet-sd21/resolve/main/control_v11p_sd21_depth.safetensors'
HED='https://huggingface.co/thibaud/controlnet-sd21/resolve/main/control_v11p_sd21_hed.safetensors'
OpenPose='https://huggingface.co/thibaud/controlnet-sd21/resolve/main/control_v11p_sd21_openposev2.safetensors'
Scribble='https://huggingface.co/thibaud/controlnet-sd21/resolve/main/control_v11p_sd21_scribble.safetensors'


if v2_Model == "All":
  for lnk_v2 in mdllnk_v2:
      download(lnk_v2, mdldir)
  clear_output()
  inf('\u2714 Done','success', '50px')

elif v2_Model == "None":
    pass
    clear_output()
    inf('\u2714 Done','success', '50px')

else:
  download(globals()[v2_Model], mdldir)
  clear_output()
  inf('\u2714 Done','success', '50px')

  #@markdown ---

Button(button_style='success', description='✔ Done', disabled=True, layout=Layout(min_width='50px'), style=But…

In [59]:
import time
import sys
import os
from pyngrok import ngrok, conf

#@markdown # Start Stable-Diffusion

Ngrok_token = "" #@param {type:"string"}
User = "" #@param {type:"string"}
Password= "" #@param {type:"string"}

auth=f"--gradio-auth {User}:{Password}"
if User == "" or Password == "":
    auth=""

# Fix for Python 3.12 compatibility and truncated script
%cd /content/gdrive/$mainpth/sd/stable-diffusion-webui/

# Ensure the model path is correctly passed
ckpt_arg = f"--ckpt \"{PATH_to_MODEL}\"" if 'PATH_to_MODEL' in locals() and PATH_to_MODEL else ""

# Setup Ngrok if token is provided
if Ngrok_token:
    conf.get_default().auth_token = Ngrok_token
    port = 7860
    public_url = ngrok.connect(port).public_url
    print(f"Ngrok Tunnel Online: {public_url}")

# Launch command with bypass for typical Colab/Python 3.12 errors
!python launch.py --share --opt-sdp-attention --disable-safe-unpickle --no-half-vae --skip-torch-cuda-test {auth} {ckpt_arg}

/content/gdrive/MyDrive/sd/stable-diffusion-webui
Python 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
Version: v1.10.1
Commit hash: 82a973c04367123ae98bd9abdf80d9eda9b910e2
Cloning Stable Diffusion into /content/gdrive/MyDrive/sd/stable-diffusion-webui/repositories/stable-diffusion-stability-ai...
Cloning into '/content/gdrive/MyDrive/sd/stable-diffusion-webui/repositories/stable-diffusion-stability-ai'...
fatal: could not read Username for 'https://github.com': No such device or address
Traceback (most recent call last):
  File "/content/gdrive/MyDrive/sd/stable-diffusion-webui/launch.py", line 48, in <module>
    main()
  File "/content/gdrive/MyDrive/sd/stable-diffusion-webui/launch.py", line 39, in main
    prepare_environment()
  File "/content/gdrive/MyDrive/sd/stable-diffusion-webui/modules/launch_utils.py", line 412, in prepare_environment
    git_clone(stable_diffusion_repo, repo_dir('stable-diffusion-stability-ai'), "Stable Diffusion", stable_diffusion_commit_hash)
  Fi

In [60]:
# ==========================================
# UNIFIED COLAB HOTFIX FOR AUTOMATIC1111/FORGE
# ==========================================

# 1. Force upgrade scikit-image and patch protobuf to resolve import errors
print("[+] Upgrading dependencies to resolve circular import conflicts...")
!pip install --upgrade -q scikit-image protobuf==3.20.3

# 2. Programmatically locate and patch 'sd_models.py' on Google Drive
import os

gdrive_models_path = "/content/gdrive/MyDrive/sd/stable-diffusion-webui/modules/sd_models.py"

if os.path.exists(gdrive_models_path):
    print(f"[+] Found sd_models.py at: {gdrive_models_path}")
    with open(gdrive_models_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Target parameter causing the load_models() TypeError
    deprecated_param = ", hash_prefix=expected_sha256"

    if deprecated_param in content:
        content = content.replace(deprecated_param, "")
        with open(gdrive_models_path, "w", encoding="utf-8") as f:
            f.write(content)
        print("[+] Successfully patched sd_models.py (deprecated 'hash_prefix' argument removed)")
    else:
        print("[*] 'hash_prefix' argument is already patched or not present in this file.")
else:
    print(f"[-] sd_models.py was not found at: {gdrive_models_path}")
    print("[*] Note: If your WebUI directory is different, please update the 'gdrive_models_path' variable above.")

[+] Upgrading dependencies to resolve circular import conflicts...
[+] Found sd_models.py at: /content/gdrive/MyDrive/sd/stable-diffusion-webui/modules/sd_models.py
[*] 'hash_prefix' argument is already patched or not present in this file.


In [61]:
import os
from pyngrok import ngrok, conf

#@markdown # Start Stable-Diffusion (Forced Python 3.10)

Ngrok_token = "" #@param {type:"string"}
User = "" #@param {type:"string"}
Password= "" #@param {type:"string"}

auth = f"--gradio-auth {User}:{Password}" if User and Password else ""

# Navigate to the WebUI directory
%cd /content/gdrive/$mainpth/sd/stable-diffusion-webui/

# Ensure the model path is correctly passed
ckpt_arg = f"--ckpt \"{PATH_to_MODEL}\"" if 'PATH_to_MODEL' in locals() and PATH_to_MODEL else ""

# Setup Ngrok if token is provided
if Ngrok_token:
    conf.get_default().auth_token = Ngrok_token
    port = 7860
    public_url = ngrok.connect(port).public_url
    print(f"Ngrok Tunnel Online: {public_url}")

# Launching with forced Python 3.10 and skipping internal environment prep
# This bypasses the pip-install loops and version checks that fail on Colab
!python3 launch.py --share --opt-sdp-attention --disable-safe-unpickle --no-half-vae --skip-torch-cuda-test --skip-python-version-check --skip-prepare-environment {auth} {ckpt_arg}

/content/gdrive/MyDrive/sd/stable-diffusion-webui
Launching Web UI with arguments: --share --opt-sdp-attention --disable-safe-unpickle --no-half-vae --skip-torch-cuda-test --skip-python-version-check --skip-prepare-environment --ckpt /content/gdrive/MyDrive/_18v2.safetensors
Traceback (most recent call last):
  File "/content/gdrive/MyDrive/sd/stable-diffusion-webui/launch.py", line 48, in <module>
    main()
  File "/content/gdrive/MyDrive/sd/stable-diffusion-webui/launch.py", line 44, in main
    start()
  File "/content/gdrive/MyDrive/sd/stable-diffusion-webui/modules/launch_utils.py", line 465, in start
    import webui
  File "/content/gdrive/MyDrive/sd/stable-diffusion-webui/webui.py", line 13, in <module>
    initialize.imports()
  File "/content/gdrive/MyDrive/sd/stable-diffusion-webui/modules/initialize.py", line 23, in imports
    import gradio  # noqa: F401
  File "/usr/local/lib/python3.10/dist-packages/gradio/__init__.py", line 3, in <module>
    import gradio._simple_temp

In [62]:
#@title Launching Stable Diffusion (Automatic Execution)
import os

# Ensure we are in the correct directory
%cd /content/gdrive/MyDrive/sd/stable-diffusion-webui/

# Define the model path verified earlier
model_path = "/content/gdrive/MyDrive/_18v2.safetensors"

# Run the WebUI using Python 3.10 to bypass 3.12 incompatibilities
# --skip-prepare-environment prevents it from attempting to overwrite our manual fixes
!python3.10 launch.py --share --opt-sdp-attention --disable-safe-unpickle --no-half-vae --skip-torch-cuda-test --skip-python-version-check --skip-prepare-environment --ckpt "{model_path}"

/content/gdrive/MyDrive/sd/stable-diffusion-webui
Launching Web UI with arguments: --share --opt-sdp-attention --disable-safe-unpickle --no-half-vae --skip-torch-cuda-test --skip-python-version-check --skip-prepare-environment --ckpt /content/gdrive/MyDrive/_18v2.safetensors
Traceback (most recent call last):
  File "/content/gdrive/MyDrive/sd/stable-diffusion-webui/launch.py", line 48, in <module>
    main()
  File "/content/gdrive/MyDrive/sd/stable-diffusion-webui/launch.py", line 44, in main
    start()
  File "/content/gdrive/MyDrive/sd/stable-diffusion-webui/modules/launch_utils.py", line 465, in start
    import webui
  File "/content/gdrive/MyDrive/sd/stable-diffusion-webui/webui.py", line 13, in <module>
    initialize.imports()
  File "/content/gdrive/MyDrive/sd/stable-diffusion-webui/modules/initialize.py", line 23, in imports
    import gradio  # noqa: F401
  File "/usr/local/lib/python3.10/dist-packages/gradio/__init__.py", line 3, in <module>
    import gradio._simple_temp

In [63]:
import socket

def check_port(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

port = 7860
if check_port(port):
    print(f"[+] A server is active on localhost:{port}")
    !ps aux | grep "launch.py" | grep -v grep
else:
    print(f"[-] No server found on localhost:{port}. The WebUI is likely offline.")

[-] No server found on localhost:7860. The WebUI is likely offline.


In [ ]:
from google.colab import output
import time

print("Checking if the WebUI is running...")
# Give the server a moment to potentially start if you just clicked run
time.sleep(2)

try:
    # This creates a direct URL to the internal 7860 port via Colab's infrastructure
    proxy_url = output.proxy_port(7860)
    print(f"\u2714 Access your WebUI via Colab Proxy: {proxy_url}")
    print("\nNote: You must keep the launch cell running for this link to work.")
except Exception as e:
    print(f"\u2718 Could not generate proxy link: {e}")

In [2]:
from google.colab.output import eval_js
try:
    # Correct way to get the external URL for a Colab local port
    url = eval_js("google.colab.kernel.proxyPort(7860)")
    print(f"\u2714 Access your WebUI via Colab Proxy: {url}")
    print("\nNote: The WebUI must be actively running in the launch cell for this link to work.")
except Exception as e:
    print(f"\u2718 Error creating proxy: {e}")

✔ Access your WebUI via Colab Proxy: https://7860-gpu-t4-s-kkb-usw4b2-1q8untvu4lnaj-b.us-west4-2.prod.colab.dev

Note: The WebUI must be actively running in the launch cell for this link to work.


In [1]:
print('Ensuring pip is available for Python 3.10 and fixing dependencies...')
# Install pip for 3.10 explicitly
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

# Apply the Pydantic/FastAPI version fix
!python3.10 -m pip install "pydantic<2.0" "fastapi<0.110.0" --upgrade --quiet

print('\u2714 Fix applied. Please re-run the Launch cell (9cc74116).')

Ensuring pip is available for Python 3.10 and fixing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [wheel]
✔ Fix applied. Please re-run the Launch cell (9cc74116).


In [64]:
print('Upgrading setuptools to resolve ImpImporter error...')
!pip install --upgrade setuptools -qq
print('setuptools upgrade complete. Please re-run the "Start Stable-Diffusion" cell.')

Upgrading setuptools to resolve ImpImporter error...
setuptools upgrade complete. Please re-run the "Start Stable-Diffusion" cell.


In [65]:
print('Reinstalling setuptools to a compatible version...')
!pip install setuptools==69.5.1 -qq
print('setuptools reinstallation complete. Please re-run the "Start Stable-Diffusion" cell.')

Reinstalling setuptools to a compatible version...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.15.12 requires appdirs>=1.4.3, which is not installed.
wandb 0.15.12 requires docker-pycreds>=0.4.0, which is not installed.
wandb 0.15.12 requires GitPython!=3.1.29,>=1.0.0, which is not installed.
wandb 0.15.12 requires pathtools, which is not installed.
wandb 0.15.12 requires psutil>=5.0.0, which is not installed.
wandb 0.15.12 requires sentry-sdk>=1.0.0, which is not installed.
wandb 0.15.12 requires setproctitle, which is not installed.
setuptools reinstallation complete. Please re-run the "Start Stable-Diffusion" cell.


In [66]:
print('Installing Python 3.10...')
!apt-get install python3.10 python3.10-venv python3.10-dev -y -qq
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!update-alternatives --set python3 /usr/bin/python3.10

# Fix pip for the new python version
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3

print('\nPython version now in use:')
!python --version

Installing Python 3.10...
E: Unmet dependencies. Try 'apt --fix-broken install' with no packages (or specify a solution).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.6 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.1.2
    Uninstalling pip-26.1.2:
      Successfully uninstalled pip-26.1.2

Python version now in use:
Python 3.10.12


In [67]:
print('Fixing Python 3.10 dependencies...')
# Install missing core components and configuration utilities
!python3 -m pip install "numpy<2" pytorch-lightning==1.9.4 gradio omegaconf einops
!python3 -m pip install https://github.com/openai/CLIP/archive/d50d76daa670286dd6cacf3bcd80b5e4823fc8e1.zip --prefer-binary
!python3 -m pip install open_clip_torch
print('Dependencies updated. Please re-run the launch cell (fe052c23).')

Fixing Python 3.10 dependencies...
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.10.2
    Uninstalling pydantic-1.10.2:
      Successfully uninstalled pydantic-1.10.2
  Using cached d50d76daa670286dd6cacf3bcd80b5e4823fc8e1.zip (4.3 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'https://github.com/openai/CLIP/archive/d50d76daa670286dd6cacf3bcd80b5e4823fc8e1.zip' when getting requirements to build wheel
Dependencies updated. Please re-run the launch cell (fe052c23).


In [68]:
import subprocess

# Mapping import names to pip package names
required_packages = {
    'omegaconf': 'omegaconf',
    'einops': 'einops',
    'gradio': 'gradio',
    'pytorch_lightning': 'pytorch-lightning==1.9.4',
    'clip': 'git+https://github.com/openai/CLIP.git',
    'open_clip': 'open_clip_torch',
    'cv2': 'opencv-python',
    'PIL': 'Pillow',
    'numpy': 'numpy<2',
    'torch': 'torch',
    'safetensors': 'safetensors',
    'piexif': 'piexif',
    'clean_fid': 'clean-fid',
    'resize_right': 'resize-right',
    'fonts': 'fonts',
    'timm': 'timm',
    'kornia': 'kornia'
}

print("Verifying dependencies inside the Python 3.10 environment...")

missing_packages = []
for imp_name, pip_name in required_packages.items():
    result = subprocess.run(['python3.10', '-c', f'import {imp_name}'], capture_output=True)
    if result.returncode != 0:
        missing_packages.append(pip_name)

if missing_packages:
    print(f"\u2718 Missing dependencies: {len(missing_packages)}")
    print(f"Installing: {', '.join(missing_packages)}")
    !python3.10 -m pip install { ' '.join(missing_packages) } --quiet
    print("\u2714 Installation complete.")
else:
    print("\u2714 All core dependencies are correctly installed in the Python 3.10 environment.")

Verifying dependencies inside the Python 3.10 environment...
✘ Missing dependencies: 1
Installing: clean-fid
✔ Installation complete.


In [69]:
print('Skipping xformers installation due to persistent incompatibility issues.')
print('Stable Diffusion WebUI can still run without xformers, though it may be slower.')

Skipping xformers installation due to persistent incompatibility issues.
Stable Diffusion WebUI can still run without xformers, though it may be slower.


In [70]:
print('Checking and installing missing dependencies: xmltodict and polygraphy...')

try:
    import xmltodict
    print('xmltodict is already installed.')
except ImportError:
    print('xmltodict not found. Installing...')
    !pip install xmltodict -qq
    print('xmltodict installed.')

try:
    import polygraphy
    print('polygraphy is already installed.')
except ImportError:
    print('polygraphy not found. Installing...')
    !pip install polygraphy -qq
    print('polygraphy installed.')

print('Dependency check and installation complete for xmltodict and polygraphy.')

Checking and installing missing dependencies: xmltodict and polygraphy...
xmltodict is already installed.
polygraphy is already installed.
Dependency check and installation complete for xmltodict and polygraphy.


In [71]:
print('Installing mmcv...')
!pip install mmcv-full -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1.0/index.html
print('mmcv installation complete.')

Installing mmcv...
Looking in links: https://download.openmmlab.com/mmcv/dist/cu121/torch2.1.0/index.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 MB 19.2 MB/s  0:00:03
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.0
    Uninstalling numpy-1.26.0:
      Successfully uninstalled numpy-1.26.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [mmcv-full]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
accelerate 1.14.0 requires psutil, which is not installed.
controlnet-aux 0.0.10 requires opencv-python-headless, which is not installed.


mmcv installation complete.


In [2]:
print('Restarting reinstallation of diffusers and accelerate...')
!pip install diffusers accelerate -U --no-deps -qq
print('\u2714 diffusers and accelerate reinstallation complete.')

Restarting reinstallation of diffusers and accelerate...
✔ diffusers and accelerate reinstallation complete.
